In [1]:
!pip install flwr torch torchvision -q
print("All packages installed successfully")

All packages installed successfully


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import copy

print("=" * 55)
print("  TrustChain — Federated Learning Simulation")
print("  Sivaranjani R | Anna University, Madurai")
print("  3 pharmacy nodes — privacy-preserving demand")
print("=" * 55)

# ── Demand forecasting neural network
class PharmacyDemandNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

# ── Generate local pharmacy dispensing data
# Each pharmacy has DIFFERENT data — stays on their device
def make_data(pharmacy_id, n=200):
    torch.manual_seed(pharmacy_id * 42)
    # 7 features = Mon-Sun dispensing history → predict next day demand
    X = torch.randn(n, 7) + pharmacy_id * 0.5
    y = X.mean(dim=1, keepdim=True) + torch.randn(n, 1) * 0.1
    return DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

# ── Local training at each pharmacy (data never leaves this function)
def local_train(model, dataloader, epochs=3):
    model = copy.deepcopy(model)
    opt = optim.Adam(model.parameters(), lr=0.005)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in dataloader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    # Only model weights returned — RAW DATA NEVER SHARED
    return model.state_dict(), len(dataloader.dataset)

# ── FedAvg: average weights from all pharmacy models
def federated_average(weight_list, sample_counts):
    total = sum(sample_counts)
    avg = {}
    for key in weight_list[0]:
        avg[key] = sum(
            w[key] * (n / total)
            for w, n in zip(weight_list, sample_counts)
        )
    return avg

# ── Evaluate global model on a pharmacy's local data
def evaluate(model, dataloader):
    loss_fn = nn.MSELoss()
    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in dataloader:
            total += loss_fn(model(xb), yb).item()
    return total / len(dataloader)

# ══ MAIN SIMULATION ═════════════════════════════════════
NUM_PHARMACIES = 3
NUM_ROUNDS     = 3

# Create pharmacy datasets (stored locally — never centralised)
pharmacies = []
for i in range(NUM_PHARMACIES):
    data = make_data(i)
    pharmacies.append(data)
    print(f"  Pharmacy {i+1} ready — "
          f"{len(data.dataset)} local dispensing records "
          f"(data stays local)")

# Initialise one global model on the server
global_model = PharmacyDemandNet()
print(f"\n  Global model initialised on server")
print(f"  Starting {NUM_ROUNDS} federated rounds...\n")

for rnd in range(1, NUM_ROUNDS + 1):
    print(f"  {'─'*50}")
    print(f"  Round {rnd} — server sends global model to pharmacies")

    local_weights = []
    local_sizes   = []

    for i, data in enumerate(pharmacies):
        # Each pharmacy trains on its LOCAL data only
        weights, n = local_train(global_model, data)
        local_weights.append(weights)
        local_sizes.append(n)
        loss = evaluate(copy.deepcopy(global_model).eval().__class__().eval()
                        if False else global_model, data)
        print(f"  Pharmacy {i+1} trained locally — "
              f"{n} records used — "
              f"raw data NOT shared")

    # Server aggregates ONLY the weight updates
    averaged = federated_average(local_weights, local_sizes)
    global_model.load_state_dict(averaged)

    # Evaluate global model
    losses = [evaluate(global_model, d) for d in pharmacies]
    avg_loss = sum(losses) / len(losses)
    print(f"  Server aggregated via FedAvg — "
          f"global loss: {avg_loss:.4f}")

print(f"\n{'='*55}")
print(f"  Federated learning complete!")
print(f"  {NUM_ROUNDS} rounds × {NUM_PHARMACIES} pharmacies")
print(f"  Zero patient data was centralised")
print(f"  Global demand model distributed back to all pharmacies")
print(f"{'='*55}")

  TrustChain — Federated Learning Simulation
  Sivaranjani R | Anna University, Madurai
  3 pharmacy nodes — privacy-preserving demand
  Pharmacy 1 ready — 200 local dispensing records (data stays local)
  Pharmacy 2 ready — 200 local dispensing records (data stays local)
  Pharmacy 3 ready — 200 local dispensing records (data stays local)

  Global model initialised on server
  Starting 3 federated rounds...

  ──────────────────────────────────────────────────
  Round 1 — server sends global model to pharmacies
  Pharmacy 1 trained locally — 200 records used — raw data NOT shared
  Pharmacy 2 trained locally — 200 records used — raw data NOT shared
  Pharmacy 3 trained locally — 200 records used — raw data NOT shared
  Server aggregated via FedAvg — global loss: 0.0450
  ──────────────────────────────────────────────────
  Round 2 — server sends global model to pharmacies
  Pharmacy 1 trained locally — 200 records used — raw data NOT shared
  Pharmacy 2 trained locally — 200 records 